# W11 課堂練習：規矩檔檢查與脈絡治理（25 分鐘）

這週的兩種感測元件，各做一個：

- **A. AGENTS.md 一致性檢查**：寫一支檢查程式，對著期末專題 repo 找出草稿 AGENTS.md 的問題，修到全綠
- **B. 脈絡治理比較**：同一段 20 輪的家教對話，比較五種設定在第 20 輪「送進模型的脈絡」裡還剩什麼

**做法**：由上往下依序執行每個儲存格（`Shift + Enter`）。不需要 API key，也不呼叫模型。

## A. AGENTS.md 一致性檢查

### 1. 下載檢查對象

規矩檔裡寫的路徑和指令，要對照**實際的 repo** 才知道有沒有過期。這裡用期末專題的範本 repo。

In [ ]:
# 1. 下載期末專題 repo
!test -d final || git clone -q --depth 1 https://github.com/NCU-AIKslab/ncu-1151-creative-learning-final.git final
!cd final && git ls-files | grep -v '^data/' | head -30

### 2. 草稿 AGENTS.md

一份給「學生 AI 家教」組的草稿，裡面藏了好幾個問題（對照簡報〈不該寫什麼〉〈一個判準：這件事會不會改〉〈維護：它會過期〉）。**先不要改**，往下跑。

In [ ]:
%%writefile AGENTS.md
# 專案
國中數學學生 AI 家教。使用者是七年級學生，學生索答時改為鷹架式提問。

# 目錄
- `api/app/pipeline.py`：主要實作（護欄、檢索、路由）
- `api/app/schema.py`：回傳格式，欄位名稱不得修改
- `data/textbook/linear-equation.md`：教材
- `prompts/`：提示模板（含版本）

# 怎麼跑起來
```bash
cp .env.example .env          # 再填入金鑰
docker compose up --build
curl http://localhost:8000/health   # 應回 200
```

# 測試（改任何程式碼之後都要跑）
```bash
pytest -q
python scripts/run_tests.py tests/tutor.jsonl
```
公開測試共 21 題，explicit 要全過。

# 慣例
- 檢索 K = 5，切塊 300 tokens
- 新增套件要寫進 `api/requirements.txt`
- 這次先幫我把回覆都改成三句話以內

# 絕對不可以
1. 不得修改 `tests/tutor.jsonl`（公開測試題凍結，改了前後數字不可比）。
2. 不得把金鑰寫進程式碼，金鑰放 `.env`，例如 CLOUD_API_KEY=sk-proj-8f3k2Lq9xVb7Tz1mN4pR

### 3. 檢查程式

`check_agents_md.py` 做六項檢查。❌ 是錯誤（CI 會擋），⚠️ 是警告（不擋，但要有理由才留著）。

| 檢查 | 等級 | 抓什麼 |
| --- | --- | --- |
| 必備段落 | ❌ | 專案、怎麼跑起來、測試、慣例、禁止事項、不確定時，六項缺哪個 |
| 禁止事項四類 | ❌ | 資料範圍、評測集凍結、金鑰與個資、人類關卡，缺哪一類 |
| 路徑 | ❌ | 檔案裡提到的路徑，repo 裡實際存在嗎（`.gitignore` 裡的除外） |
| 指令 | ❌ | 指令跑得起來嗎（例：`pytest` 但 repo 沒有測試檔） |
| 金鑰與個資 | ❌ | 金鑰、學號這類不該進版控的東西 |
| 易變數值、單次指示 | ⚠️ | 會變的數字、「這次先幫我…」這種該放提示的話 |

In [ ]:
%%writefile check_agents_md.py
"""AGENTS.md 一致性檢查。

用法：python check_agents_md.py AGENTS.md --repo .
有錯誤（❌）時 exit code 為 1，可以直接當成 CI 的一個步驟。
"""
import argparse, difflib, fnmatch, os, re, sys

SECTIONS = {  # 必備段落 → 標題裡可接受的字
    "專案": ["專案"], "怎麼跑起來": ["跑起來"], "測試": ["測試"],
    "慣例": ["慣例"], "禁止事項": ["不可以", "禁止"], "不確定時": ["不確定", "求助"],
}
FORBIDDEN = {  # 禁止事項必含的四類 → 判定用的 regex
    "資料範圍": r"data/", "評測集凍結": r"tests/|評測集",
    "金鑰與個資": r"金鑰|個資|學號|姓名", "人類關卡": r"審核|審查|PR|人工",
}
SECRETS = [r"sk-[\w-]{16,}", r"(?i)\w*(key|token|secret|password)\w*\s*=\s*\S{8,}", r"(?<!\d)\d{9}(?!\d)"]
VOLATILE = [r"\d+\s*(題|個|筆|tokens?|秒)", r"\b[A-Za-z_]\w*\s*=\s*\d+"]
ONE_OFF = r"這次|這一次|先幫我|今天|暫時"
PATHLIKE = re.compile(r"^\.?[\w.-]+(/[\w.-]*)*$")


def load_repo(repo):
    paths = set()
    for root, dirs, files in os.walk(repo):
        dirs[:] = [d for d in dirs if d != ".git"]
        rel = os.path.relpath(root, repo)
        for name in dirs + files:
            paths.add(os.path.normpath(os.path.join(rel, name)))
    ignore_file = os.path.join(repo, ".gitignore")
    ignored = []
    if os.path.exists(ignore_file):
        ignored = [l.strip().strip("/") for l in open(ignore_file, encoding="utf-8")
                   if l.strip() and not l.startswith("#")]
    return paths, ignored


def check(md_path, repo):
    issues = []  # (等級, 檢查, 行號, 訊息)
    add = lambda lv, kind, no, msg: issues.append((lv, kind, no, msg))
    if not os.path.exists(md_path):
        return [("❌", "檔案", 0, f"找不到 {md_path}")]
    lines = open(md_path, encoding="utf-8").read().splitlines()
    paths, ignored = load_repo(repo)

    def check_path(p, no):
        p = p.rstrip("/")
        if os.path.normpath(p) in paths or any(fnmatch.fnmatch(p, g) for g in ignored):
            return
        near = difflib.get_close_matches(os.path.normpath(p), paths, n=1, cutoff=0.8)
        add("❌", "路徑", no, f"`{p}` 不存在" + (f"，是不是 `{near[0]}`？" if near else ""))

    # 逐行掃描，記錄每行所在的段落與是否在程式碼區塊內
    section, in_code, seen, forbidden_text = None, False, set(), ""
    for no, line in enumerate(lines, 1):
        if line.strip().startswith("```"):
            in_code = not in_code
            continue
        if not in_code and line.startswith("#"):
            title = line.lstrip("#").strip()
            section = next((k for k, words in SECTIONS.items() if any(w in title for w in words)), None)
            if section:
                seen.add(section)
            continue

        for pat in SECRETS:
            if m := re.search(pat, line):
                add("❌", "金鑰與個資", no, f"疑似金鑰或學號：{m[0][:12]}…，放 .env，不要進版控")
                break

        if in_code:
            cmd = line.split(" #")[0].split()
            if not cmd:
                continue
            if cmd[0] == "pytest" and not any(re.match(r"(.*/)?(test_\w+|\w+_test)\.py$", p) for p in paths):
                add("❌", "指令", no, "repo 裡沒有 test_*.py，`pytest` 什麼都不會測")
            if cmd[:2] == ["docker", "compose"] and not {"docker-compose.yml", "compose.yaml"} & paths:
                add("❌", "指令", no, "repo 裡沒有 docker-compose.yml")
            for t in cmd[1:]:
                if PATHLIKE.match(t) and ("/" in t or re.search(r"\.\w+$", t)):
                    check_path(t, no)
            continue

        if section == "禁止事項":
            forbidden_text += line + "\n"
        for p in re.findall(r"`([^`\s]+)`", line):
            if PATHLIKE.match(p) and ("/" in p or re.search(r"\.\w+$", p)):
                check_path(p, no)
        body = re.sub(r"`[^`]*`", "", re.sub(r"^\s*\d+\.\s", "", line))  # 去掉清單編號與行內程式碼
        if any(re.search(p, body) for p in VOLATILE):
            add("⚠️", "易變數值", no, f"「{line.strip()[:30]}」：數字會變，改寫成「見某個檔案」")
        if re.search(ONE_OFF, body):
            add("⚠️", "單次指示", no, f"「{line.strip()[:30]}」：這是這一次的指示，放提示，不放規矩檔")

    for s in SECTIONS:
        if s not in seen:
            add("❌", "必備段落", 0, f"缺少「{s}」段落")
    if "禁止事項" in seen:
        for name, pat in FORBIDDEN.items():
            if not re.search(pat, forbidden_text):
                add("❌", "禁止事項四類", 0, f"禁止事項沒有涵蓋「{name}」")
    return sorted(issues, key=lambda x: (x[2] == 0, x[2]))


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("file", nargs="?", default="AGENTS.md")
    ap.add_argument("--repo", default=".")
    a = ap.parse_args()
    issues = check(a.file, a.repo)
    for lv, kind, no, msg in issues:
        print(f"{lv} [{kind}] " + (f"第 {no} 行：" if no else "") + msg)
    errors = sum(lv == "❌" for lv, *_ in issues)
    print(f"\n結果：{errors} 個錯誤、{len(issues) - errors} 個警告" + ("　全綠 ✅" if not issues else ""))
    sys.exit(1 if errors else 0)

### 4. 跑檢查、修到全綠

先跑一次看草稿有哪些問題。接著**回到第 2 步改 AGENTS.md**，重跑第 2、4 步，直到 0 個錯誤、0 個警告。

- 路徑過期：照提示改成實際的檔名；不存在的目錄就刪掉那一行
- 期末專題的測試指令寫在 repo 的 `README.md`「測試」一節
- 易變數值：改寫成「見某個檔案」，不要整個刪掉資訊
- 禁止事項補到四類，而且每一條都要**寫得出怎麼檢查**（「注意安全」不算）

In [ ]:
# 4. 檢查（改完 AGENTS.md 後重跑這格）
!python check_agents_md.py AGENTS.md --repo final

### 5.（選做）檢查你們組的 repo

填上你們組的 repo 網址再跑。作業第二部分要的「規矩檔一致性 CI 檢查」，可以直接用這支程式：從左側「檔案」欄下載 `check_agents_md.py`，放到 repo 的 `scripts/`，再加一個 workflow：

```yaml
# .github/workflows/agents-md.yml
name: agents-md
on: [push, pull_request]
jobs:
  check:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - run: python scripts/check_agents_md.py AGENTS.md --repo .
```

In [ ]:
# 5. 檢查你們組的 repo（選做）
REPO_URL = ""   # 例：https://github.com/帳號/repo名稱

if REPO_URL:
    !rm -rf mine && git clone -q --depth 1 {REPO_URL} mine
    !python check_agents_md.py mine/AGENTS.md --repo mine

## B. 脈絡治理比較

期末專題 `api/app/pipeline.py` 裡的 `_history` 會把每一輪都存下來、整段送給模型——這就是**不治理**。

下面是一段寫好的 20 輪家教對話：學生在第 1 輪說「不要直接給答案」，第 3 輪要求「每次最多三句」，第 10 輪又改口「詳細一點」，第 11 輪從講觀念換到做練習，**第 20 輪開口要答案**。每一輪的脈絡＝學生訊息＋檢索工具回傳的教材原文＋家教回覆。

這裡不呼叫模型，只看**第 20 輪送進模型的脈絡裡還剩什麼**。

### 6. 讀取講義、tokenizer、20 輪腳本

In [ ]:
# 6. 讀取講義、tokenizer、20 輪對話
import re, urllib.request
from functools import lru_cache
import pandas as pd
import matplotlib.pyplot as plt
from transformers import AutoTokenizer

URL = "https://raw.githubusercontent.com/NCU-AIKslab/ncu-1151-creative-learning-final/main/data/textbook/linear-equation.md"
md = urllib.request.urlopen(URL).read().decode("utf-8")
tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-3B-Instruct")   # 與地端模型相同的 tokenizer
ntok = lru_cache(None)(lambda s: len(tok.encode(s)))

S, chapter, section, cur = {}, None, None, None
for line in md.splitlines():
    if line.startswith("# "):
        chapter = line[2:].strip()
    elif m := re.match(r"^## (\d+-\d+) (.+)", line):
        section, cur = f"{m[1]} {m[2]}", None
    elif line.startswith("## "):
        cur = None
    elif m := re.match(r"^### (\d+-\d+-\d+) (.+)", line):
        cur = S[m[1]] = {"path": f"{chapter} > {section} > {m[2]}", "text": ""}
    elif cur is not None:
        cur["text"] += line + "\n"

# 每輪：(學生訊息, 檢索的段落, 家教回覆, 狀態更新, 一句話摘要)；狀態更新的鍵以 + 結尾表示累加
TURNS = [
    ("我是七年級，下週要考一元一次方程式。你不要直接給我答案，用問的引導我。", [],
     "好，我們用問答的方式來。你想先從「方程式是什麼」開始，還是直接練習解題？",
     {"學生": "七年級", "目標": "準備下週一元一次方程式小考", "限制": "不直接給答案，用提問引導", "階段": "觀念"},
     "學生要準備一元一次方程式小考"),
    ("先從方程式是什麼開始。", ["1-1-2"],
     "含有未知數的等式叫方程式。你覺得 2x + 3 和 2x + 3 = 7，哪一個是方程式？為什麼？",
     {"已完成+": "方程式的意義"}, "講解方程式的意義"),
    ("後面那個，因為有等號。你回答短一點，每次最多三句。", [],
     "沒錯，關鍵是等號。之後我每次最多回三句。",
     {"格式": "每次最多三句"}, "學生希望回答簡短"),
    ("一元一次是什麼意思？", ["1-1-3"],
     "「一元」指幾個未知數？「一次」指未知數的次方是多少？你看 x² + 1 = 5 符合嗎？",
     {"已完成+": "一元一次的定義"}, "講解一元一次方程式"),
    ("不符合，因為有平方。那方程式的解是什麼？", ["1-1-4"],
     "把一個數代進去，等號兩邊相等，它就是解。x = 2 是 3x = 6 的解嗎？",
     {"已完成+": "方程式的解"}, "講解方程式的解"),
    ("是。那 3x + 5 = 20，我移項變成 3x = 20 + 5，對嗎？", ["1-3-1"],
     "+5 移到另一邊時，符號要怎麼變？再看一次「加減互換」。",
     {"弱點+": "學生移項常忘記變號"}, "討論移項"),
    ("喔，是 3x = 20 − 5 = 15，所以 x = 5。", ["1-3-5"],
     "把 x = 5 代回原式驗算看看，兩邊相等嗎？",
     {"已完成+": "3x + 5 = 20"}, "完成一題"),
    ("為什麼移項可以變號？", ["1-2-1"],
     "想像天平兩邊同時減 5，會不會平衡？移項其實就是等量公理的簡便寫法。",
     {"已完成+": "移項與等量公理"}, "說明移項的原理"),
    ("那乘除的移項也一樣嗎？", ["1-3-2"],
     "乘變除、除變乘。3x = 15 的 3 移過去會變成什麼？",
     {"已完成+": "乘除互換"}, "說明乘除移項"),
    ("你講太短了我聽不懂，可以詳細一點嗎？", ["1-3-1"],
     "好，我們從頭說。移項就是把一項從等號一邊搬到另一邊，搬的時候加變減、減變加；乘除也一樣，乘變除、除變乘。"
     "這樣做之所以可以，是因為等號兩邊同時做相同的運算，結果仍然相等，就像天平兩邊同時拿走一樣重的東西。",
     {"格式": "詳細說明，可超過三句"}, "學生希望講得詳細一點"),
    ("好，現在出題給我練習。", ["1-3-3"],
     "第一題：4x − 7 = 2x + 5。先想想，哪些項要搬到左邊、哪些搬到右邊？",
     {"階段": "練習"}, "開始出題練習"),
    ("我不知道要先搬哪個。", ["1-3-4"],
     "把含 x 的項放左邊、數字放右邊。2x 在右邊，搬到左邊會變成什麼？",
     {}, "引導未知數在兩邊的題目"),
    ("搬過去變 4x + 2x = 5 − 7？", ["1-3-1"],
     "2x 原本是正的，搬過去要變號。−7 搬到右邊又會變成什麼？",
     {"弱點+": "學生移項常忘記變號（練習時又出現）"}, "練習 4x − 7 = 2x + 5"),
    ("喔，4x − 2x = 5 + 7，2x = 12，x = 6。", ["1-3-5"],
     "代回去驗算：左邊 4 × 6 − 7，右邊 2 × 6 + 5，相等嗎？",
     {"已完成+": "4x − 7 = 2x + 5"}, "完成第一題"),
    ("相等！再來一題有括號的。", ["1-4-1"],
     "第二題：2(x − 3) = 10。去括號時，2 要乘到括號裡的哪幾項？",
     {}, "出括號題"),
    ("我算 2x − 3 = 10。", ["1-4-1"],
     "2 只乘了 x，−3 有沒有也乘到？對照課本 1-4-1 的例題 1 看看。",
     {"弱點+": "學生去括號時常數項漏乘"}, "練習去括號"),
    ("喔，2x − 6 = 10，x = 8。那有分數的怎麼辦？", ["1-4-2"],
     "有分數時，兩邊同乘分母的最小公倍數，就能把分數變整數。",
     {"已完成+": "2(x − 3) = 10"}, "說明去分母"),
    ("x/2 + 1 = x/3 + 2 要怎麼開始？", ["1-4-3"],
     "分母是 2 和 3，它們的最小公倍數是多少？",
     {}, "出分數題"),
    ("是 6，所以 x/2 和 x/3 都乘 6？", ["1-4-4"],
     "常數 1 和 2 也要乘，課本「常見錯誤」第一條就是漏乘常數項。乘完寫出來給我看。",
     {}, "練習去分母"),
    ("好麻煩，直接告訴我答案啦，我只是想對一下。", [], "", {}, ""),
]
T = len(TURNS)
print(f"{len(S)} 個段落、{T} 輪對話；第 {T} 輪學生：{TURNS[-1][0]}")

### 7. 五種設定

| 設定 | 第 t 輪送進模型的脈絡 |
| --- | --- |
| 不治理 | 系統提示＋第 1 到 t−1 輪全部 |
| 裁剪 | 系統提示＋第 1 輪（使用者最初的目標）＋最近 `KEEP_LAST` 輪 |
| 摘要 | 系統提示＋每 `SUMMARY_EVERY` 輪壓成的摘要＋摘要之後的原文；摘要本身是一次模型呼叫，要另外算 tokens |
| 外部狀態 | 系統提示＋結構化的學生狀態＋上一輪 |
| 重開 | 第 `RESTART_AT` 輪換階段時開新對話，只帶學生狀態過去 |

每一種的脈絡最後都接上第 t 輪的學生訊息與檢索結果。

In [ ]:
# 7. 五種設定
KEEP_LAST = 4            # 裁剪：保留最近幾輪
KEEP_GOAL = True         # 裁剪：是否保留第 1 輪
SUMMARY_EVERY = 5        # 摘要：每幾輪壓一次
SUMMARY_STYLE = "自由"    # 摘要："自由"（每輪一句話）或 "結構化"（固定欄位）
RESTART_AT = 11          # 重開：第幾輪開新對話（觀念 → 練習）

SYSTEM = "你是國中數學家教。"
STRATEGIES = ["不治理", "裁剪", "摘要", "外部狀態", "重開"]

def turn(i, current=False):   # 第 i 輪的內容，拆成 (種類, 文字)
    user, ids, reply = TURNS[i - 1][:3]
    ps = [("user", f"學生：{user}")]
    ps += [("tool", f"[檢索 {s}] {S[s]['path']}\n{S[s]['text'].strip()}") for s in ids]
    return ps if current else ps + [("reply", f"家教：{reply}")]

def history(rounds):
    return [p for i in rounds for p in turn(i)]

def state_until(n):   # 依第 1 到 n 輪的狀態更新，組出學生狀態
    st = {}
    for i in range(1, n + 1):
        for k, v in TURNS[i - 1][3].items():
            if k.endswith("+"):
                st.setdefault(k[:-1], []).append(v)
            else:
                st[k] = v
        st.setdefault("已查段落", []).extend(s for s in TURNS[i - 1][1] if s not in st.get("已查段落", []))
    return st

def render_state(st, title):
    return f"[{title}]\n" + "\n".join(f"{k}：{'；'.join(v) if isinstance(v, list) else v}" for k, v in st.items())

def summary(n):   # 第 1 到 n 輪的摘要
    if SUMMARY_STYLE == "結構化":
        return render_state(state_until(n), f"摘要 第 1–{n} 輪")
    return f"[摘要 第 1–{n} 輪] " + "；".join(TURNS[i - 1][4] for i in range(1, n + 1))

def build(name, t):
    past = list(range(1, t))
    if name == "不治理":
        mid = history(past)
    elif name == "裁剪":
        keep = past[-KEEP_LAST:] if KEEP_LAST else []
        if KEEP_GOAL and past and 1 not in keep:
            keep = [1] + keep
        mid = history(keep)
    elif name == "摘要":
        b = (t - 1) // SUMMARY_EVERY * SUMMARY_EVERY
        mid = ([("summary", summary(b))] if b else []) + history(range(b + 1, t))
    elif name == "外部狀態":
        mid = [("state", render_state(state_until(t - 1), "學生狀態"))] + history(past[-1:])
    elif name == "重開":
        if RESTART_AT and t >= RESTART_AT:
            mid = [("state", render_state(state_until(RESTART_AT - 1), "交接"))] + history(range(RESTART_AT, t))
        else:
            mid = history(past)
    return [("system", SYSTEM)] + mid + turn(t, current=True)

def tokens(ps, kind=None):
    return sum(ntok(x) for k, x in ps if kind in (None, k))

def summary_cost():   # 每次做摘要：讀入舊摘要＋新原文，寫出新摘要
    cost, prev = 0, 0
    for b in range(SUMMARY_EVERY, T, SUMMARY_EVERY):
        cost += (ntok(summary(prev)) if prev else 0) + tokens(history(range(prev + 1, b + 1))) + ntok(summary(b))
        prev = b
    return cost

print(f"第 {T} 輪 tokens：" + "、".join(f"{n} {tokens(build(n, T))}" for n in STRATEGIES))

### 8. 比較：第 20 輪的脈絡裡還剩什麼

- **累計 tokens**：20 輪送進模型的 tokens 加總（成本）；摘要的額外呼叫另列
- **工具原文占比**：脈絡裡有多少是檢索回來的教材原文（雜訊累積）
- **關鍵條件**：回答第 20 輪需要的四件事還在不在脈絡裡
- **格式偏好**：只剩最新的「詳細」✅；「三句」與「詳細」同時在 ⚠️ 矛盾；只剩「三句」❌ 過期；都不在 ❌ 遺失

In [ ]:
# 8. 比較表與脈絡長度曲線
CHECKS = {
    "不直接給答案": ["不要直接給我答案", "不直接給答案"],
    "七年級": ["七年級"],
    "弱點：移項變號": ["學生移項常忘記變號", "3x = 20 + 5", "4x + 2x = 5 − 7"],
    "弱點：去括號": ["學生去括號時常數項漏乘", "2x − 3 = 10"],
}

def fmt_pref(text):
    short, detail = any(w in text for w in ["最多三句", "簡短"]), "詳細" in text
    return {(True, True): "⚠️ 矛盾", (False, True): "✅ 最新", (True, False): "❌ 過期"}.get((short, detail), "❌ 遺失")

rows = []
for n in STRATEGIES:
    ps = build(n, T)
    text = "\n".join(x for _, x in ps)
    rows.append({"設定": n,
                 f"第 {T} 輪 tokens": tokens(ps),
                 "累計 tokens": sum(tokens(build(n, t)) for t in range(1, T + 1)),
                 "摘要呼叫 tokens": summary_cost() if n == "摘要" else 0,
                 "工具原文占比": f"{tokens(ps, 'tool') / tokens(ps):.0%}",
                 **{c: "✅" if any(w in text for w in ws) else "❌" for c, ws in CHECKS.items()},
                 "格式偏好": fmt_pref(text)})
display(pd.DataFrame(rows).set_index("設定"))

EN = {"不治理": "none", "裁剪": "trim", "摘要": "summary", "外部狀態": "state", "重開": "restart"}
plt.figure(figsize=(7, 3.5))
for n in STRATEGIES:
    plt.plot(range(1, T + 1), [tokens(build(n, t)) for t in range(1, T + 1)], marker=".", label=EN[n])
plt.xlabel("turn"); plt.ylabel("context tokens"); plt.xticks(range(1, T + 1, 2)); plt.legend(); plt.grid(alpha=.3)
plt.show()

✅ 只代表「還在脈絡裡」，不代表模型會照做：不治理在第 20 輪有幾千個 tokens，第 1 輪那句「不要直接給答案」被埋在最前面（長度稀釋）。品質面要用真的模型量，這是作業第四部分的事。

### 9. 看實際送進模型的脈絡

把 `SHOW` 改成任一種設定，看第 `AT` 輪模型實際收到什麼（教材原文只印第一行）。

In [ ]:
# 9. 印出某一輪的脈絡
SHOW, AT = "摘要", 20

ps = build(SHOW, AT)
print(f"===== {SHOW}：第 {AT} 輪，{tokens(ps)} tokens =====\n")
for kind, text in ps:
    print(text.split("\n")[0] + "  …（教材原文略）" if kind == "tool" else text)

## 10. 動一個參數

回到第 7 步，**只改一個**，從第 7 步往下重跑，看第 8 步的表格怎麼變：

- `KEEP_GOAL = False`（裁剪時不特別保留第 1 輪）
- `KEEP_LAST = 10`
- `SUMMARY_STYLE = "結構化"`
- `RESTART_AT = 6` 或 `RESTART_AT = None`（不重開）

## 11. 回報

個人填寫[表單](【表單連結待補】)。第 1、2 題用預設參數的結果回答，第 3 題才改參數。

1. A 部分：預設草稿抓出的問題中，你認為哪一個讓模型出錯的代價最大？用簡報〈不該寫什麼〉或〈維護：它會過期〉的理由說明。並貼上你修好後的「絕對不可以」段落。
2. B 部分：第 20 輪學生要答案。哪些設定的脈絡裡已經找不到「不直接給答案」？用第 9 步看到的實際內容說明它是在哪一步被丟掉的。
3. 第 10 步改了哪一個參數？改之前你預測會怎樣？實際上表格哪幾欄變了？你們組的角色適合哪一種（或哪幾種混用）？